<a href="https://colab.research.google.com/github/ajtony3000/Autonomic-Complexity-Under-a-Cognitive-Motor-Dual-Task-Paradigm/blob/main/autonomic_complexity_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

hrv_sampen_lmem_pipeline.py
100%
# -*- coding: utf-8 -*-
"""
Autonomic Complexity Under a Cognitive-Motor Dual-Task Paradigm:
A Hierarchical Linear Mixed-Effects Modeling of Heart Rate Sample Entropy

Authors:
- Aswin Tony Joseph (Government Medical College, Thiruvananthapuram, Kerala, India)
- Dr. Rahul Venugopal (Centre for Consciousness Studies, Department of Neurophysiology, NIMHANS, Bengaluru, India)

Dataset: OpenNeuro ds007554 ("CMx7-MM")
- 30 Recruited Participants | 3 Sessions (ses-01, ses-02, ses-03) | 7 Hierarchical Tasks
- Sampling Rate: 1,882.45 Hz (Delsys Trigno wireless ECG)

Pipeline Stages:
1. Optional AWS S3 BIDS Data Download (OpenNeuro ds007554)
2. Automated ECG Preprocessing, Nabian (2018) R-Peak Detection, RR Artifact Filtering & EDR Extraction
3. Validated Relational Merge (KSS & Cognitive Load) with Nomenclature & Cardinality Audits
4. CONSORT-Style Subject & Recording Attrition Flowchart Generation (Figure 1)
5. Descriptive Statistics, Normality/Homoscedasticity Screening & Univariate Baseline Analyses
6. Primary Three-Level Nested LMEM (Task Effects & Baseline Fatigue; re_formula="~1" + Session vc_formula)
7. Secondary Three-Level Nested LMEM (Within-Subject Scaled Cognitive Load & Task-Collinearity Diagnostic)
8. Complete-Case (No Imputation) & Outlier-Free Sensitivity Analyses
9. Multi-Domain Comparative LMEM Pipeline (7 Target Physiological Metrics) & Publication Visualizations
"""

import logging
import os
import traceback
import warnings
from pathlib import Path

import matplotlib
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from patsy import dmatrices
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.sm_exceptions import ConvergenceWarning

# Suppress non-critical convergence and pandas warnings
warnings.filterwarnings("ignore")
warnings.simplefilter("ignore", ConvergenceWarning)
warnings.simplefilter("ignore", UserWarning)

# Matplotlib guards for headless execution and high-density time series
matplotlib.rcParams["agg.path.chunksize"] = 10000
matplotlib.use("Agg")

# Canonical ordering of the 7 hierarchical tasks (Easiest to Hardest)
ORDERED_TASKS = [
    "passivemotor",
    "activemotor",
    "motorimagery",
    "mentalarithmetic",
    "nback",
    "nbackarithmetic",
    "full",
]

SAMPLING_RATE = 1882.45


# ==============================================================================
# STAGE 1: OPTIONAL AWS S3 DOWNLOAD HELPER (OPENNEURO ds007554)
# ==============================================================================
def download_openneuro_ds007554(base_dir: Path = Path("./ds007554_data")) -> dict:
    """
    Downloads raw ECG recordings and phenotype TSVs from OpenNeuro ds007554 via AWS S3.
    Requires `awscli` installed (`pip install awscli`).
    """
    physio_dir = base_dir / "physio"
    pheno_dir = base_dir / "phenotype"
    physio_dir.mkdir(parents=True, exist_ok=True)
    pheno_dir.mkdir(parents=True, exist_ok=True)

    os.system(
        f'aws s3 sync --no-sign-request s3://openneuro.org/ds007554 "{physio_dir}" '
        '--exclude "*" --include "*_recording-ECG_physio.tsv.gz"'
    )
    os.system(
        f'aws s3 sync --no-sign-request s3://openneuro.org/ds007554 "{pheno_dir}" '
        '--exclude "*" --include "participants.tsv" '
        '--include "phenotype/cognitive_load.tsv" --include "phenotype/kss_values.tsv"'
    )
    return {"physio_dir": physio_dir, "pheno_dir": pheno_dir}


# ==============================================================================
# STAGE 2: ECG SIGNAL PREPROCESSING & HRV / EDR EXTRACTION PIPELINE
# ==============================================================================
def run_ecg_preprocessing_pipeline(
    data_dir: Path, results_dir: Path
) -> pd.DataFrame:
    """
    Processes raw BIDS ECG recordings (1882.45 Hz) using NeuroKit2:
    1. Lead polarity verification (nk.ecg_invert)
    2. Bandpass filtering & initial QRS detection (nk.ecg_process)
    3. Hard Signal Quality Index gate (Mean SQI >= 0.50)
    4. Artifact-corrected R-peak detection via Nabian et al. (2018)
    5. Physiological RR bounding (300-1500 ms; >10% bad intervals flagged 'POOR')
    6. ECG-Derived Respiration (EDR) extraction (mean_resp_rate, sd_resp_rate)
    7. Multi-domain HRV feature extraction (HRV_SampEn, m=2, r=0.2*SD)
    """
    import neurokit2 as nk

    qc_dir = results_dir / "QC_Plots"
    results_dir.mkdir(parents=True, exist_ok=True)
    qc_dir.mkdir(parents=True, exist_ok=True)

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
        datefmt="%H:%M:%S",
        handlers=[
            logging.StreamHandler(),
            logging.FileHandler(results_dir / "pipeline_run_log.txt"),
        ],
    )
    log = logging.getLogger(__name__)

    filelist = sorted(data_dir.glob("**/*_recording-ECG_physio.tsv.gz"))
    if not filelist:
        raise FileNotFoundError(f"No ECG TSV.gz files found under: {data_dir}")

    log.info("Found %d ECG file(s) to process.", len(filelist))
    masterlist = []

    for file_no, file_path in enumerate(filelist, start=1):
        fname = file_path.name
        parts = fname.split("_")

        sub_id = next((p for p in parts if p.startswith("sub-")), "Unknown")
        session_id = next((p for p in parts if p.startswith("ses-")), "Unknown")
        task_name = next(
            (p for p in parts if p.startswith("task-")), "Unknown"
        ).replace("task-", "")

        log.info(
            "── [%d/%d] Processing: %s | %s | %s",
            file_no,
            len(filelist),
            sub_id,
            session_id,
            task_name,
        )

        try:
            df = pd.read_csv(file_path, sep="\t", compression="gzip", header=None)
            ecg_raw_signal = df.iloc[:, 0].values

            # 1. Automated polarity verification
            ecg_signal, is_inverted = nk.ecg_invert(
                ecg_raw_signal, sampling_rate=SAMPLING_RATE, show=False
            )

            # 2. Primary ECG filtering
            signals, ecg_info = nk.ecg_process(
                ecg_signal, sampling_rate=SAMPLING_RATE
            )

            # 3. Signal Quality Index (SQI) gate
            quality_scores = nk.ecg_quality(
                signals["ECG_Clean"],
                rpeaks=ecg_info["ECG_R_Peaks"],
                sampling_rate=SAMPLING_RATE,
            )
            mean_sqi = float(np.mean(quality_scores))

            if mean_sqi < 0.5:
                log.warning(
                    "SQI %.3f < 0.5 — signal too noisy, skipping %s.",
                    mean_sqi,
                    fname,
                )
                continue

            # 4. Nabian (2018) R-peak refinement with artifact correction
            _, peaks_info = nk.ecg_peaks(
                signals["ECG_Clean"],
                correct_artifacts=True,
                method="nabian2018",
                sampling_rate=SAMPLING_RATE,
            )
            ecg_info["ECG_R_Peaks"] = peaks_info["ECG_R_Peaks"]
            signals["ECG_R_Peaks"] = 0
            signals.loc[ecg_info["ECG_R_Peaks"], "ECG_R_Peaks"] = 1

            # 5. Physiological RR interval filtering (300 - 1500 ms)
            rr_ms = np.diff(ecg_info["ECG_R_Peaks"]) / SAMPLING_RATE * 1000.0
            bad_mask = (rr_ms < 300.0) | (rr_ms > 1500.0)
            n_bad = int(np.sum(bad_mask))
            pct_bad = 100.0 * n_bad / len(rr_ms) if len(rr_ms) > 0 else 0.0
            data_quality_flag = "POOR" if pct_bad > 10.0 else "OK"

            # 6. ECG-Derived Respiration (EDR)
            ecg_rate_signal = nk.ecg_rate(
                peaks_info,
                sampling_rate=SAMPLING_RATE,
                desired_length=len(ecg_signal),
            )
            edr_signal = nk.ecg_rsp(ecg_rate_signal, sampling_rate=SAMPLING_RATE)
            rsp_signals, _ = nk.rsp_process(
                edr_signal, sampling_rate=SAMPLING_RATE
            )
            mean_rsp_rate = float(np.mean(rsp_signals["RSP_Rate"]))
            sd_rsp_rate = float(np.std(rsp_signals["RSP_Rate"], ddof=1))

            # 7. Multi-domain HRV computation
            hrv_df = nk.hrv(ecg_info, sampling_rate=SAMPLING_RATE, show=True)
            hrv_fig = plt.gcf()
            hrv_fig.set_size_inches(15, 8)
            hrv_fig.savefig(
                qc_dir / f"{sub_id}_{session_id}_{task_name}_hrv_summary.png",
                dpi=150,
            )
            plt.close("all")

            # 8. Metadata tagging
            hrv_df["Subject"] = sub_id
            hrv_df["Session"] = session_id
            hrv_df["Task"] = task_name
            hrv_df["is_inverted"] = is_inverted
            hrv_df["mean_resp_rate"] = mean_rsp_rate
            hrv_df["sd_resp_rate"] = sd_rsp_rate
            hrv_df["n_rpeaks"] = len(ecg_info["ECG_R_Peaks"])
            hrv_df["mean_sqi"] = round(mean_sqi, 4)
            hrv_df["pct_rr_bad"] = round(pct_bad, 2)
            hrv_df["data_quality_flag"] = data_quality_flag

            # 9. Save ECG morphological quality check plot
            nk.ecg_plot(signals, info=ecg_info)
            qual_fig = plt.gcf()
            qual_fig.set_size_inches(15, 8)
            qual_fig.savefig(
                qc_dir / f"{sub_id}_{session_id}_{task_name}_quality_check.png",
                dpi=150,
            )
            plt.close("all")

            masterlist.append(hrv_df)

        except Exception:
            log.error("FAILED for %s:\n%s", fname, traceback.format_exc())
            continue

    if not masterlist:
        raise RuntimeError("No ECG files were processed successfully.")

    master_df = pd.concat(masterlist, axis=0, ignore_index=True)
    id_cols = [
        "Subject",
        "Session",
        "Task",
        "data_quality_flag",
        "mean_sqi",
        "n_rpeaks",
        "pct_rr_bad",
        "is_inverted",
        "mean_resp_rate",
        "sd_resp_rate",
    ]
    other_cols = [c for c in master_df.columns if c not in id_cols]
    master_df = master_df[id_cols + other_cols]

    output_csv = results_dir / "Autonomic_Complexity_Results_CCS_Validated.csv"
    master_df.to_csv(output_csv, index=False)
    log.info("Saved validated HRV table to: %s", output_csv)
    return master_df


# ==============================================================================
# STAGE 3: HARDENED PHENOTYPE MERGE WITH CARDINALITY & MATCH ASSERTIONS
# ==============================================================================
def merge_behavioral_phenotypes(
    df_hrv: pd.DataFrame,
    df_demo: pd.DataFrame,
    df_kss_values: pd.DataFrame,
    df_cognitive_load: pd.DataFrame,
) -> pd.DataFrame:
    """
    Filters QC-passed finite HRV_SampEn observations and merges demographics,
    KSS sleepiness scores, and post-task Cognitive Load ratings with strict
    nomenclature normalization and m:1 merge validation.
    """
    # Filter for 'OK' RR quality and finite Sample Entropy
    clean_df = df_hrv[df_hrv["data_quality_flag"] == "OK"].copy()
    clean_df["HRV_SampEn"] = clean_df["HRV_SampEn"].replace(
        [np.inf, -np.inf], np.nan
    )
    clean_df = clean_df.dropna(subset=["HRV_SampEn"]).copy()

    # Merge demographics
    demo_clean = df_demo.rename(columns={"participant_id": "Subject"}).copy()
    final_df = pd.merge(
        clean_df, demo_clean, on="Subject", how="left", validate="m:1"
    )

    # 1. Prepare KSS (wide -> long, normalize 'sess-' -> 'ses-')
    df_kss_long = df_kss_values.rename(columns={"participant_id": "Subject"})
    df_kss_long = pd.melt(
        df_kss_long,
        id_vars=["Subject"],
        var_name="Session_KSS",
        value_name="KSS_Score",
    )
    df_kss_long["Session"] = (
        df_kss_long["Session_KSS"]
        .str.split("_")
        .str[0]
        .str.replace("sess-", "ses-", regex=False)
        .str.strip()
    )
    df_kss_clean = df_kss_long[
        ["Subject", "Session", "KSS_Score"]
    ].drop_duplicates(subset=["Subject", "Session"])

    # 2. Prepare Cognitive Load (wide -> long, strip underscores/hyphens, remap aliases)
    df_cogload_long = df_cognitive_load.rename(
        columns={"participant_id": "Subject"}
    )
    df_cogload_long = pd.melt(
        df_cogload_long,
        id_vars=["Subject"],
        var_name="Session_Task",
        value_name="Cognitive_Load_Score",
    )
    df_cogload_long["Session"] = (
        df_cogload_long["Session_Task"]
        .str.split("_")
        .str[0]
        .str.replace("sess-", "ses-", regex=False)
        .str.strip()
    )
    df_cogload_long["Task_Raw"] = df_cogload_long["Session_Task"].apply(
        lambda x: "_".join(x.split("_")[1:]).lower().strip()
    )
    df_cogload_long["Task"] = (
        df_cogload_long["Task_Raw"]
        .str.replace("_", "", regex=False)
        .str.replace("-", "", regex=False)
        .str.replace(" ", "", regex=False)
    )

    task_remap = {
        "arithmetic": "mentalarithmetic",
        "mentalarithmetic": "mentalarithmetic",
        "passivemotor": "passivemotor",
        "activemotor": "activemotor",
        "motorimagery": "motorimagery",
        "nback": "nback",
        "nbackarithmetic": "nbackarithmetic",
        "full": "full",
    }
    df_cogload_long["Task"] = df_cogload_long["Task"].replace(task_remap)

    hrv_tasks = set(final_df["Task"].unique())
    cog_tasks = set(df_cogload_long["Task"].unique())
    unmatched = hrv_tasks - cog_tasks
    assert len(unmatched) == 0, f"Unmatched HRV tasks found: {unmatched}"

    df_cogload_clean = df_cogload_long[
        ["Subject", "Session", "Task", "Cognitive_Load_Score"]
    ].drop_duplicates(subset=["Subject", "Session", "Task"])

    # 3. Validated Grand Merge
    master_df = final_df.drop(
        columns=[c for c in final_df.columns if "sess-" in c], errors="ignore"
    )
    master_df = pd.merge(
        master_df,
        df_kss_clean,
        on=["Subject", "Session"],
        how="left",
        validate="m:1",
        indicator="_kss_merge",
    )
    master_df = pd.merge(
        master_df,
        df_cogload_clean,
        on=["Subject", "Session", "Task"],
        how="left",
        validate="m:1",
        indicator="_cog_merge",
    )

    kss_match_rate = (master_df["_kss_merge"] == "both").mean() * 100.0
    cog_match_rate = (master_df["_cog_merge"] == "both").mean() * 100.0
    assert kss_match_rate == 100.0, f"KSS key mismatch: {kss_match_rate:.1f}%"
    assert cog_match_rate == 100.0, f"CogLoad key mismatch: {cog_match_rate:.1f}%"

    master_df = master_df.drop(columns=["_kss_merge", "_cog_merge"])
    return master_df


# ==============================================================================
# STAGE 4: CONSORT-STYLE SUBJECT & RECORDING ATTRITION FLOWCHART (FIGURE 1)
# ==============================================================================
def plot_consort_attrition_flowchart(
    raw_csv_path: Path = Path("Autonomic_Complexity_Results_CCS_Validated.csv"),
    merged_csv_path: Path = Path("Autonomic_Complexity_Cognitive_Load_Merged.csv"),
    output_fig_path: str = "Figure_1_Subject_Attrition_Flowchart.png",
) -> None:
    """
    Generates the 6-stage CONSORT-style flowchart detailing recording and
    subject attrition from N=30 (n=630 expected) to N=29 (n=450 LMEM analysed).
    """
    N_THEORETICAL_SUBJ = 30
    N_THEORETICAL_OBS = 630
    N_RAW_FILES = 467

    if raw_csv_path.exists():
        df_hrv = pd.read_csv(raw_csv_path)
        n_subj_raw = df_hrv["Subject"].nunique()
        n_sqi_pass = len(df_hrv)
        n_sqi_drop = N_RAW_FILES - n_sqi_pass

        df_ok = df_hrv[df_hrv["data_quality_flag"] == "OK"].copy()
        n_rr_pass = len(df_ok)
        n_rr_drop = n_sqi_pass - n_rr_pass
        n_subj_ok = df_ok["Subject"].nunique()

        df_ok["HRV_SampEn"] = df_ok["HRV_SampEn"].replace(
            [np.inf, -np.inf], np.nan
        )
        df_clean = df_ok.dropna(subset=["HRV_SampEn"]).copy()
        n_sampen_pass = len(df_clean)
        n_sampen_drop = n_rr_pass - n_sampen_pass
        n_subj_sampen = df_clean["Subject"].nunique()
    else:
        n_sqi_pass, n_sqi_drop, n_subj_raw = 467, 0, 30
        n_rr_pass, n_rr_drop, n_subj_ok = 464, 3, 30
        n_sampen_pass, n_sampen_drop, n_subj_sampen = 456, 8, 29

    if merged_csv_path.exists():
        df_merged = pd.read_csv(merged_csv_path)
        df_lmem = df_merged.dropna(
            subset=["HRV_SampEn", "sex", "Task", "Session", "KSS_Score"]
        ).copy()
        n_lmem_pass = len(df_lmem)
        n_kss_drop = n_sampen_pass - n_lmem_pass
        n_subj_final = df_lmem["Subject"].nunique()
    else:
        n_lmem_pass, n_kss_drop, n_subj_final = 450, 6, 29

    n_missing_raw = N_THEORETICAL_OBS - N_RAW_FILES

    fig, ax = plt.subplots(figsize=(12.0, 11.5), dpi=300)
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 11.2)
    ax.axis("off")

    def draw_box(
        ax,
        center_x,
        center_y,
        width,
        height,
        title,
        body,
        bg_color="#F8F9FA",
        border_color="#2B4C7E",
        title_color="#1A365D",
    ):
        x = center_x - width / 2
        y = center_y - height / 2
        rect = patches.FancyBboxPatch(
            (x, y),
            width,
            height,
            boxstyle="round,pad=0.08,rounding_size=0.15",
            facecolor=bg_color,
            edgecolor=border_color,
            linewidth=1.8,
            zorder=3,
        )
        ax.add_patch(rect)
        ax.text(
            center_x,
            center_y + height * 0.20,
            title,
            ha="center",
            va="center",
            fontsize=10.0,
            fontweight="bold",
            color=title_color,
            zorder=4,
        )
        ax.text(
            center_x,
            center_y - height * 0.16,
            body,
            ha="center",
            va="center",
            fontsize=9.0,
            color="#222222",
            linespacing=1.35,
            zorder=4,
        )

    def draw_arrow(ax, x1, y1, x2, y2):
        ax.annotate(
            "",
            xy=(x2, y2),
            xytext=(x1, y1),
            arrowprops=dict(
                arrowstyle="-|>,head_width=0.35,head_length=0.5",
                color="#333333",
                lw=1.6,
            ),
            zorder=2,
        )

    X_MAIN, X_EXCL = 3.0, 7.65
    W_MAIN, W_EXCL, H_BOX = 4.6, 4.2, 1.10
    y_levels = [10.2, 8.35, 6.5, 4.65, 2.8, 0.95]

    draw_box(
        ax,
        X_MAIN,
        y_levels[0],
        W_MAIN,
        H_BOX,
        title="1. Theoretical Study Design (ds007554)",
        body=f"N = {N_THEORETICAL_SUBJ} Participants | 3 Sessions × 7 Tasks\nExpected Recordings: n = {N_THEORETICAL_OBS}",
        bg_color="#EBF3FA",
        border_color="#2B4C7E",
    )
    draw_box(
        ax,
        X_MAIN,
        y_levels[1],
        W_MAIN,
        H_BOX,
        title="2. Raw ECG Files Available",
        body=f"Extracted from OpenNeuro BIDS Repository\nAvailable Recordings: n = {N_RAW_FILES} (N = {n_subj_raw} subjects)",
    )
    draw_box(
        ax,
        X_EXCL,
        (y_levels[0] + y_levels[1]) / 2,
        W_EXCL,
        0.95,
        title="Excluded: Missing Raw Data / Telemetry",
        body=f"n = {n_missing_raw} recordings missing in repository\nWireless ECG telemetry capture failed / skipped runs",
        bg_color="#FFF5F5",
        border_color="#C53030",
        title_color="#9B2C2C",
    )
    draw_box(
        ax,
        X_MAIN,
        y_levels[2],
        W_MAIN,
        H_BOX,
        title="3. Signal Quality Index (SQI) Filter",
        body=f"Passed Hard Threshold (Mean SQI ≥ 0.50)\nRetained: n = {n_sqi_pass} recordings (N = {n_subj_raw} subjects)",
    )
    draw_box(
        ax,
        X_EXCL,
        (y_levels[1] + y_levels[2]) / 2,
        W_EXCL,
        0.95,
        title="Excluded: Severe Signal Noise",
        body=f"n = {n_sqi_drop} recordings skipped in pipeline\n(Mean NeuroKit2 ECG SQI < 0.50)",
        bg_color="#FFF5F5",
        border_color="#C53030",
        title_color="#9B2C2C",
    )
    draw_box(
        ax,
        X_MAIN,
        y_levels[3],
        W_MAIN,
        H_BOX,
        title="4. Physiological RR Interval Filter",
        body=f"Passed Artifact Filter (≤ 10% Bad RR Intervals)\nRetained: n = {n_rr_pass} recordings (N = {n_subj_ok} subjects)",
    )
    draw_box(
        ax,
        X_EXCL,
        (y_levels[2] + y_levels[3]) / 2,
        W_EXCL,
        0.95,
        title="Excluded: Ectopic / Artifact Beats",
        body=f"n = {n_rr_drop} flagged as 'POOR' quality\n(> 10% of RR intervals outside 300–1500 ms)",
        bg_color="#FFF5F5",
        border_color="#C53030",
        title_color="#9B2C2C",
    )
    draw_box(
        ax,
        X_MAIN,
        y_levels[4],
        W_MAIN,
        H_BOX,
        title="5. Valid Finite Entropy (Merged Dataset)",
        body=f"Finite HRV_SampEn Retained in Master CSV\nRetained: n = {n_sampen_pass} recordings (N = {n_subj_sampen} subjects)",
    )
    draw_box(
        ax,
        X_EXCL,
        (y_levels[3] + y_levels[4]) / 2,
        W_EXCL,
        1.02,
        title="Excluded: Non-Finite Sample Entropy",
        body=f"n = {n_sampen_drop} recordings yielded Infinite/NaN HRV_SampEn\n(All remaining sub-023 recordings lost; N: 30 → 29)",
        bg_color="#FFF5F5",
        border_color="#C53030",
        title_color="#9B2C2C",
    )
    draw_box(
        ax,
        X_MAIN,
        y_levels[5],
        W_MAIN,
        H_BOX,
        title="6. Final Analytical Cohort (Primary Task LMEM)",
        body=f"Complete Behavioral Covariates (KSS & Demographics)\nFinal Analysed: n = {n_lmem_pass} obs (N = {n_subj_final} subjects)",
        bg_color="#E6F4EA",
        border_color="#1E7E34",
        title_color="#155724",
    )
    draw_box(
        ax,
        X_EXCL,
        (y_levels[4] + y_levels[5]) / 2,
        W_EXCL,
        1.02,
        title="Excluded: Missing Behavioral Covariate (KSS)",
        body=f"n = {n_kss_drop} recordings dropped prior to LMEM fit\n(sub-024 Session 1 lacked KSS score; Sessions 2–3 retained)",
        bg_color="#FFF5F5",
        border_color="#C53030",
        title_color="#9B2C2C",
    )

    for i in range(len(y_levels) - 1):
        y_top = y_levels[i] - H_BOX / 2
        y_bot = y_levels[i + 1] + H_BOX / 2
        y_mid = (y_levels[i] + y_levels[i + 1]) / 2
        draw_arrow(ax, X_MAIN, y_top, X_MAIN, y_bot)
        draw_arrow(ax, X_MAIN, y_mid, X_EXCL - W_EXCL / 2, y_mid)

    plt.suptitle(
        "CONSORT-Style Flowchart of Subject & ECG Recording Attrition (ds007554)",
        fontsize=13.5,
        fontweight="bold",
        y=0.97,
        color="#111111",
    )
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(output_fig_path, dpi=300, bbox_inches="tight")
    plt.close("all")


# ==============================================================================
# STAGE 5: VARIANCE DECOMPOSITION HELPER (NAKAGAWA R² & 3-LEVEL ICC)
# ==============================================================================
def compute_lmem_variance_metrics(result_obj) -> dict:
    """
    Computes 3-level variance decomposition (Subject trait, nested Session block,
    Fixed effects, Residual error), Nakagawa's Marginal/Conditional R²,
    Intraclass Correlation Coefficients (ICC), and Shapiro-Wilk residual normality.
    """
    var_subject = (
        0.0 if result_obj.cov_re.empty else float(result_obj.cov_re.iloc[0, 0])
    )
    var_session = (
        float(result_obj.vcomp.sum())
        if (hasattr(result_obj, "vcomp") and result_obj.vcomp is not None)
        else 0.0
    )
    var_random = var_subject + var_session
    var_resid = float(result_obj.scale)

    linear_pred = result_obj.model.exog @ result_obj.fe_params
    var_fixed = float(np.var(linear_pred))
    var_total = var_fixed + var_random + var_resid

    r2_marginal = var_fixed / var_total
    r2_conditional = (var_fixed + var_random) / var_total

    icc_total = var_random / (var_random + var_resid)
    icc_subject = var_subject / (var_random + var_resid)
    icc_session = var_session / (var_random + var_resid)

    w_stat, p_shapiro = stats.shapiro(result_obj.resid)

    return {
        "VarF": var_fixed,
        "VarR_subject": var_subject,
        "VarR_session": var_session,
        "VarE": var_resid,
        "VarT": var_total,
        "R2m": r2_marginal,
        "R2c": r2_conditional,
        "ICC_total": icc_total,
        "ICC_subject": icc_subject,
        "ICC_session": icc_session,
        "Shapiro_W": float(w_stat),
        "Shapiro_p": float(p_shapiro),
        "LogLikelihood": float(result_obj.llf),
    }


# ==============================================================================
# STAGE 6: PRIMARY TASK LMEM & COMPLETE-CASE SENSITIVITY ANALYSIS
# ==============================================================================
def run_primary_task_lmem_suite(master_df: pd.DataFrame) -> dict:
    """
    Fits the Primary Three-Level Nested Task LMEM (N=29, n=450) with explicit
    Subject random intercepts (re_formula="~1") and Session variance components,
    plus the Complete-Case (No Imputation) sensitivity model (N=18, n=297).
    """
    print("\n" + "=" * 75)
    print(" PRIMARY THREE-LEVEL NESTED TASK LMEM (N = 29, Observations = 450)")
    print("=" * 75)

    task_df = master_df.dropna(
        subset=["HRV_SampEn", "sex", "Task", "Session", "KSS_Score"]
    ).copy()

    task_df["age_numeric"] = pd.to_numeric(task_df["age"], errors="coerce")
    task_df["age_missing"] = task_df["age_numeric"].isna().astype(int)
    mean_obs_age = task_df.loc[task_df["age_missing"] == 0, "age_numeric"].mean()
    task_df["age_c"] = (task_df["age_numeric"] - mean_obs_age).fillna(0.0)
    task_df["kss_c"] = task_df["KSS_Score"] - task_df["KSS_Score"].mean()
    task_df["sex_male"] = (task_df["sex"] == "M").astype(int)

    formula_task = (
        "HRV_SampEn ~ C(Task, Treatment('passivemotor')) + "
        "age_c + age_missing + sex_male + kss_c"
    )

    model_task = smf.mixedlm(
        formula=formula_task,
        data=task_df,
        groups=task_df["Subject"],
        re_formula="~1",  # Explicit Subject Random Intercept
        vc_formula={"Session_Block": "0 + C(Session)"},
    )
    result_task = model_task.fit(method="powell")
    print(result_task.summary())

    metrics_task = compute_lmem_variance_metrics(result_task)
    print("\n--- Primary Task Model Variance & ICC Partitioning ---")
    for k, v in metrics_task.items():
        print(f"  {k:15s}: {v:.4f}")

    # Multicollinearity VIF Check
    _, X_mat = dmatrices(formula_task, data=task_df, return_type="dataframe")
    vif_df = pd.DataFrame(
        {
            "Variable": X_mat.columns,
            "VIF": [
                variance_inflation_factor(X_mat.values, i)
                for i in range(X_mat.shape[1])
            ],
        }
    )
    print("\n--- Primary Task Model VIF Check ---")
    print(vif_df[vif_df["Variable"] != "Intercept"].to_string(index=False))

    # Complete-Case Sensitivity Model (No Age Imputation, N = 18, n = 297)
    print("\n" + "=" * 75)
    print(" PRIMARY TASK COMPLETE-CASE SENSITIVITY MODEL (N = 18, Obs = 297)")
    print("=" * 75)
    cc_df = task_df.dropna(subset=["age_numeric"]).copy()
    cc_df["age_c"] = cc_df["age_numeric"] - cc_df["age_numeric"].mean()
    cc_df["kss_c"] = cc_df["KSS_Score"] - cc_df["KSS_Score"].mean()

    formula_cc = (
        "HRV_SampEn ~ C(Task, Treatment('passivemotor')) + "
        "age_c + sex_male + kss_c"
    )
    result_cc = smf.mixedlm(
        formula=formula_cc,
        data=cc_df,
        groups=cc_df["Subject"],
        re_formula="~1",
        vc_formula={"Session_Block": "0 + C(Session)"},
    ).fit(method="powell")
    print(result_cc.summary())

    metrics_cc = compute_lmem_variance_metrics(result_cc)
    return {
        "task_df": task_df,
        "result_task": result_task,
        "metrics_task": metrics_task,
        "result_cc": result_cc,
        "metrics_cc": metrics_cc,
    }


# ==============================================================================
# STAGE 7: SECONDARY LMEM (WITHIN-SUBJECT SCALED COGNITIVE LOAD & SENSITIVITY)
# ==============================================================================
def run_secondary_cognitive_load_suite(master_df: pd.DataFrame) -> dict:
    """
    Fits the Secondary Three-Level LMEM using within-subject z-scaled subjective
    Cognitive Load (`cogload_within_z`):
    - Model 2A: Primary within-subject coupling across the task gradient (N=29, n=449)
    - Model 2B: Task-controlled collinearity diagnostic (C(Task) + cogload_within_z)
    - Model A (Sensitivity): Full cohort outlier-removed (|z_resid| <= 3 SD; n=446)
    - Model B (Sensitivity): Complete-case no age imputation (N=18, n=297)
    - Model C (Sensitivity): Complete-case + outlier-removed (N=18, n=295)
    """
    print("\n" + "=" * 75)
    print(" SECONDARY LMEM 2A: WITHIN-SUBJECT SCALED COGNITIVE LOAD (N = 29, n = 449)")
    print("=" * 75)

    load_df = master_df.dropna(
        subset=[
            "HRV_SampEn",
            "Cognitive_Load_Score",
            "sex",
            "Task",
            "Session",
            "KSS_Score",
        ]
    ).copy()

    load_df["age_numeric"] = pd.to_numeric(load_df["age"], errors="coerce")
    load_df["age_missing"] = load_df["age_numeric"].isna().astype(int)
    mean_obs_age = load_df.loc[load_df["age_missing"] == 0, "age_numeric"].mean()
    load_df["age_c"] = (load_df["age_numeric"] - mean_obs_age).fillna(0.0)

    load_df["Cognitive_Load_Score"] = pd.to_numeric(
        load_df["Cognitive_Load_Score"], errors="coerce"
    )
    subj_mean = load_df.groupby("Subject")["Cognitive_Load_Score"].transform(
        "mean"
    )
    subj_sd = load_df.groupby("Subject")["Cognitive_Load_Score"].transform("std")
    load_df["cogload_within_z"] = (
        load_df["Cognitive_Load_Score"] - subj_mean
    ) / subj_sd

    load_df["kss_c"] = load_df["KSS_Score"] - load_df["KSS_Score"].mean()
    load_df["sex_male"] = (load_df["sex"] == "M").astype(int)

    formula_2a = (
        "HRV_SampEn ~ cogload_within_z + age_c + age_missing + sex_male + kss_c"
    )
    result_2a = smf.mixedlm(
        formula=formula_2a,
        data=load_df,
        groups=load_df["Subject"],
        re_formula="~1",
        vc_formula={"Session_Block": "0 + C(Session)"},
    ).fit(method="powell")
    print(result_2a.summary())

    metrics_2a = compute_lmem_variance_metrics(result_2a)
    print("\n--- Model 2A Variance & ICC Partitioning ---")
    for k, v in metrics_2a.items():
        print(f"  {k:15s}: {v:.4f}")

    # Model 2B: Task-Controlled Confounding / Collinearity Diagnostic
    print("\n" + "=" * 75)
    print(" MODEL 2B: TASK-CONTROLLED COLLINEARITY DIAGNOSTIC")
    print("=" * 75)
    formula_2b = (
        "HRV_SampEn ~ cogload_within_z + C(Task, Treatment('passivemotor')) + "
        "age_c + age_missing + sex_male + kss_c"
    )
    result_2b = smf.mixedlm(
        formula=formula_2b,
        data=load_df,
        groups=load_df["Subject"],
        re_formula="~1",
        vc_formula={"Session_Block": "0 + C(Session)"},
    ).fit(method="powell")
    print(result_2b.summary())

    _, X_2b = dmatrices(formula_2b, data=load_df, return_type="dataframe")
    vif_2b = pd.DataFrame(
        {
            "Variable": X_2b.columns,
            "VIF": [
                variance_inflation_factor(X_2b.values, i)
                for i in range(X_2b.shape[1])
            ],
        }
    )
    print("\n--- Collinearity VIF Diagnostic (Task + cogload_within_z) ---")
    print(vif_2b[vif_2b["Variable"] != "Intercept"].to_string(index=False))

    # Sensitivity Model A: Full Cohort Outlier-Removed (|z_resid| > 3 SD)
    resid_2a = result_2a.resid
    z_resid_2a = (resid_2a - resid_2a.mean()) / resid_2a.std()
    outlier_mask = np.abs(z_resid_2a) > 3.0
    robust_df = load_df[~outlier_mask].copy()

    r_mean = robust_df.groupby("Subject")["Cognitive_Load_Score"].transform(
        "mean"
    )
    r_sd = robust_df.groupby("Subject")["Cognitive_Load_Score"].transform("std")
    robust_df["cogload_within_z"] = (
        robust_df["Cognitive_Load_Score"] - r_mean
    ) / r_sd
    r_age_mean = robust_df.loc[
        robust_df["age_missing"] == 0, "age_numeric"
    ].mean()
    robust_df["age_c"] = (robust_df["age_numeric"] - r_age_mean).fillna(0.0)
    robust_df["kss_c"] = robust_df["KSS_Score"] - robust_df["KSS_Score"].mean()

    result_2a_robust = smf.mixedlm(
        formula=formula_2a,
        data=robust_df,
        groups=robust_df["Subject"],
        re_formula="~1",
        vc_formula={"Session_Block": "0 + C(Session)"},
    ).fit(method="powell")

    # Sensitivity Model B: Complete-Case (No Imputation, N = 18, n = 297)
    ni_df = load_df.dropna(subset=["age_numeric"]).copy()
    ni_mean = ni_df.groupby("Subject")["Cognitive_Load_Score"].transform("mean")
    ni_sd = ni_df.groupby("Subject")["Cognitive_Load_Score"].transform("std")
    ni_df["cogload_within_z"] = (ni_df["Cognitive_Load_Score"] - ni_mean) / ni_sd
    ni_df["age_c"] = ni_df["age_numeric"] - ni_df["age_numeric"].mean()
    ni_df["kss_c"] = ni_df["KSS_Score"] - ni_df["KSS_Score"].mean()

    formula_ni = "HRV_SampEn ~ cogload_within_z + age_c + sex_male + kss_c"
    result_2a_ni = smf.mixedlm(
        formula=formula_ni,
        data=ni_df,
        groups=ni_df["Subject"],
        re_formula="~1",
        vc_formula={"Session_Block": "0 + C(Session)"},
    ).fit(method="powell")

    # Sensitivity Model C: Complete-Case + Outlier-Removed (N = 18, n = 295)
    resid_ni = result_2a_ni.resid
    z_resid_ni = (resid_ni - resid_ni.mean()) / resid_ni.std()
    rni_df = ni_df[np.abs(z_resid_ni) <= 3.0].copy()
    rni_mean = rni_df.groupby("Subject")["Cognitive_Load_Score"].transform(
        "mean"
    )
    rni_sd = rni_df.groupby("Subject")["Cognitive_Load_Score"].transform("std")
    rni_df["cogload_within_z"] = (
        rni_df["Cognitive_Load_Score"] - rni_mean
    ) / rni_sd
    rni_df["age_c"] = rni_df["age_numeric"] - rni_df["age_numeric"].mean()
    rni_df["kss_c"] = rni_df["KSS_Score"] - rni_df["KSS_Score"].mean()

    result_2a_ni_robust = smf.mixedlm(
        formula=formula_ni,
        data=rni_df,
        groups=rni_df["Subject"],
        re_formula="~1",
        vc_formula={"Session_Block": "0 + C(Session)"},
    ).fit(method="powell")

    return {
        "load_df": load_df,
        "result_2a": result_2a,
        "metrics_2a": metrics_2a,
        "result_2b": result_2b,
        "result_2a_robust": result_2a_robust,
        "result_2a_ni": result_2a_ni,
        "result_2a_ni_robust": result_2a_ni_robust,
    }


# ==============================================================================
# STAGE 8: MULTI-DOMAIN 7-METRIC COMPARATIVE LMEM SUITE (TABLES S4 & S5)
# ==============================================================================
def run_multidomain_comparative_suite(master_df: pd.DataFrame) -> pd.DataFrame:
    """
    Runs identical 3-level nested LMEMs across all 7 physiological targets:
    HRV_RMSSD, HRV_SDNN, HRV_SD1SD2, HRV_CSI, HRV_DFA_alpha1, mean_resp_rate, HRV_SampEn.
    """
    target_metrics = [
        "HRV_RMSSD",
        "HRV_SDNN",
        "HRV_SD1SD2",
        "HRV_CSI",
        "HRV_DFA_alpha1",
        "mean_resp_rate",
        "HRV_SampEn",
    ]
    summary_rows = []

    for metric in target_metrics:
        if metric not in master_df.columns:
            continue
        tdf = master_df.dropna(
            subset=[metric, "sex", "Task", "Session", "KSS_Score"]
        ).copy()
        tdf[metric] = pd.to_numeric(tdf[metric], errors="coerce")
        tdf = tdf.replace([np.inf, -np.inf], np.nan).dropna(subset=[metric]).copy()

        tdf["age_numeric"] = pd.to_numeric(tdf["age"], errors="coerce")
        tdf["age_missing"] = tdf["age_numeric"].isna().astype(int)
        mean_age = tdf.loc[tdf["age_missing"] == 0, "age_numeric"].mean()
        tdf["age_c"] = (tdf["age_numeric"] - mean_age).fillna(0.0)
        tdf["kss_c"] = tdf["KSS_Score"] - tdf["KSS_Score"].mean()
        tdf["sex_male"] = (tdf["sex"] == "M").astype(int)

        formula = (
            f"{metric} ~ C(Task, Treatment('passivemotor')) + "
            "age_c + age_missing + sex_male + kss_c"
        )
        res = smf.mixedlm(
            formula=formula,
            data=tdf,
            groups=tdf["Subject"],
            re_formula="~1",
            vc_formula={"Session_in_Subj": "0 + C(Session)"},
        ).fit(method="powell")

        m_stats = compute_lmem_variance_metrics(res)
        m_stats["Metric"] = metric
        m_stats["N_Subjects"] = tdf["Subject"].nunique()
        m_stats["N_Obs"] = len(tdf)
        summary_rows.append(m_stats)

    comp_df = pd.DataFrame(summary_rows)
    print("\n" + "=" * 75)
    print(" MULTI-DOMAIN 7-METRIC COMPARATIVE VARIANCE & R-SQUARED SUMMARY")
    print("=" * 75)
    print(
        comp_df[
            [
                "Metric",
                "N_Subjects",
                "N_Obs",
                "R2m",
                "R2c",
                "ICC_total",
                "ICC_subject",
                "ICC_session",
                "Shapiro_W",
                "Shapiro_p",
            ]
        ]
        .round(4)
        .to_string(index=False)
    )
    return comp_df


# ==============================================================================
# STAGE 9: PUBLICATION-READY VISUALIZATIONS (SPAGHETTI, FOREST, DIAGNOSTICS)
# ==============================================================================
def generate_publication_plots(
    task_df: pd.DataFrame, result_task, result_load
) -> None:
    """
    Generates corrected publication figures:
    - Primary LMEM Residual Diagnostics (Figure 6)
    - Fixed-Effects Forest Plot (Figure 7)
    - Task x Session Heatmap (Figure 8)
    - Corrected 7-Task Within-Subject Spaghetti Trajectory Plot (Supp Figure S3)
    - Secondary Cognitive Load LMEM Residual Diagnostics (Supp Figure S2)
    """
    # 1. Primary Task LMEM Diagnostics (Figure 6)
    residuals = result_task.resid
    fitted = result_task.fittedvalues
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), dpi=300)
    stats.probplot(residuals, dist="norm", plot=axes[0])
    axes[0].set_title("Q-Q Plot (Normality)", fontweight="bold")
    axes[1].scatter(fitted, residuals, alpha=0.5, color="steelblue")
    axes[1].axhline(0, color="darkred", linestyle="--", linewidth=2)
    axes[1].set_xlabel("Predicted Entropy")
    axes[1].set_ylabel("Residuals")
    axes[1].set_title("Residuals vs. Fitted", fontweight="bold")
    axes[2].hist(residuals, bins=20, edgecolor="black", color="lightblue")
    axes[2].set_xlabel("Residuals")
    axes[2].set_title("Residual Distribution", fontweight="bold")
    plt.tight_layout()
    plt.savefig("Figure_6_LMEM_Diagnostics.png", dpi=300, bbox_inches="tight")
    plt.close("all")

    # 2. Corrected 7-Task Spaghetti Plot (Supp Figure S3)
    fig, ax = plt.subplots(figsize=(12, 7), dpi=300)
    spaghetti_data = (
        task_df.groupby(["Subject", "Task"])["HRV_SampEn"].mean().reset_index()
    )
    for subject in spaghetti_data["Subject"].unique():
        sub_d = (
            spaghetti_data[spaghetti_data["Subject"] == subject]
            .set_index("Task")
            .reindex(ORDERED_TASKS)
        )
        ax.plot(
            sub_d.index,
            sub_d["HRV_SampEn"],
            marker="o",
            alpha=0.3,
            linewidth=1.5,
            color="gray",
        )
    group_mean = (
        spaghetti_data.groupby("Task")["HRV_SampEn"]
        .mean()
        .reindex(ORDERED_TASKS)
    )
    ax.plot(
        group_mean.index,
        group_mean.values,
        marker="D",
        markersize=8,
        linewidth=4,
        color="darkred",
        label="Group Average Trend",
        zorder=10,
    )
    ax.set_xlabel("Task Condition (Easiest to Hardest)", fontsize=13, fontweight="bold")
    ax.set_ylabel("HRV Sample Entropy (Complexity)", fontsize=13, fontweight="bold")
    ax.set_title(
        "Within-Subject Trajectories of Autonomic Entropy Collapse",
        fontsize=15,
        fontweight="bold",
    )
    ax.legend(fontsize=11, loc="upper right")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig("Supp_Figure_S3_Spaghetti_SampEn.png", dpi=300, bbox_inches="tight")
    plt.close("all")

    # 3. Forest Plot of Task Fixed Effects (Figure 7)
    params = result_task.params
    conf = result_task.conf_int()
    conf["coef"] = params
    task_stats = conf[conf.index.str.contains("Task")].copy()
    task_stats["Task_Name"] = task_stats.index.map(
        lambda x: x.replace("C(Task, Treatment('passivemotor'))[T.", "").replace(
            "]", ""
        )
    )
    fig, ax = plt.subplots(figsize=(10, 6), dpi=300)
    y_pos = np.arange(len(task_stats))
    effects = task_stats["coef"]
    ci_lower, ci_upper = task_stats[0], task_stats[1]
    ax.errorbar(
        effects,
        y_pos,
        xerr=[effects - ci_lower, ci_upper - effects],
        fmt="o",
        markersize=9,
        capsize=6,
        capthick=2,
        linewidth=2.5,
        color="darkblue",
    )
    ax.axvline(
        0,
        color="red",
        linestyle="--",
        linewidth=2,
        label="No Difference (Passive Baseline)",
    )
    ax.set_yticks(y_pos)
    ax.set_yticklabels(task_stats["Task_Name"].values, fontsize=11)
    ax.set_xlabel(
        "Difference in Entropy vs. Passive Motor (95% CI)",
        fontsize=13,
        fontweight="bold",
    )
    ax.set_title(
        "Forest Plot: Effect of Tasks on HRV Complexity",
        fontsize=15,
        fontweight="bold",
    )
    ax.legend(fontsize=11, loc="upper right")
    ax.grid(alpha=0.3, axis="x")
    plt.tight_layout()
    plt.savefig("Figure_7_ForestPlot.png", dpi=300, bbox_inches="tight")
    plt.close("all")

    # 4. Task x Session Heatmap (Figure 8)
    heatmap_data = task_df.pivot_table(
        values="HRV_SampEn", index="Task", columns="Session", aggfunc="mean"
    ).reindex(ORDERED_TASKS)
    fig, ax = plt.subplots(figsize=(8, 6), dpi=300)
    sns.heatmap(
        heatmap_data,
        annot=True,
        fmt=".3f",
        cmap="coolwarm",
        cbar_kws={"label": "Mean Sample Entropy"},
        linewidths=0.5,
        ax=ax,
    )
    ax.set_xlabel("Testing Session", fontsize=13, fontweight="bold")
    ax.set_ylabel("Task Condition", fontsize=13, fontweight="bold")
    ax.set_title(
        "Mean HRV Sample Entropy by Task and Session",
        fontsize=15,
        fontweight="bold",
    )
    plt.tight_layout()
    plt.savefig("Figure_8_Heatmap_Task_Session.png", dpi=300, bbox_inches="tight")
    plt.close("all")

    # 5. Secondary Cognitive Load Residual Diagnostics (Supp Figure S2)
    fitted_load = result_load.fittedvalues
    resid_load = result_load.resid
    std_res_load = (resid_load - resid_load.mean()) / resid_load.std()
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), dpi=300)
    sns.residplot(
        x=fitted_load,
        y=std_res_load,
        lowess=True,
        scatter_kws={"alpha": 0.5, "color": "steelblue"},
        line_kws={"color": "darkred", "lw": 2},
        ax=axes[0],
    )
    axes[0].set_title(
        "Homoscedasticity: Residuals vs. Fitted", fontsize=13, fontweight="bold"
    )
    axes[0].set_xlabel("Fitted Values (Predicted Entropy)")
    axes[0].set_ylabel("Standardized Residuals")
    axes[0].axhline(0, color="black", linestyle="--", lw=1)
    sm.qqplot(
        std_res_load,
        line="45",
        fit=True,
        markerfacecolor="steelblue",
        markeredgecolor="steelblue",
        alpha=0.5,
        ax=axes[1],
    )
    axes[1].set_title(
        "Normality: Q-Q Plot of Residuals", fontsize=13, fontweight="bold"
    )
    sns.despine()
    plt.tight_layout()
    plt.savefig(
        "Supp_Figure_S2_Diagnostic_Plots_CognitiveLoad.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close("all")


# ==============================================================================
# MAIN EXECUTION ENTRY POINT
# ==============================================================================
if __name__ == "__main__":
    merged_path = Path("Autonomic_Complexity_Cognitive_Load_Merged.csv")
    if not merged_path.exists():
        merged_path = Path("/content/Autonomic_Complexity_Cognitive_Load_Merged.csv")

    if merged_path.exists():
        print(f"[INFO] Loading merged dataset from: {merged_path}")
        master_df = pd.read_csv(merged_path)

        # 1. Generate CONSORT Attrition Flowchart (Figure 1)
        plot_consort_attrition_flowchart(merged_csv_path=merged_path)

        # 2. Run Primary Three-Level Nested Task LMEM & Sensitivity Suite
        primary_outputs = run_primary_task_lmem_suite(master_df)

        # 3. Run Secondary Within-Subject Scaled Cognitive Load LMEM & Sensitivity Suite
        secondary_outputs = run_secondary_cognitive_load_suite(master_df)

        # 4. Run 7-Metric Multi-Domain Comparative Suite
        multidomain_df = run_multidomain_comparative_suite(master_df)

        # 5. Generate Publication Figures
        generate_publication_plots(
            task_df=primary_outputs["task_df"],
            result_task=primary_outputs["result_task"],
            result_load=secondary_outputs["result_2a"],
        )
        print("\n[SUCCESS] Full statistical & visualization pipeline completed.")
    else:
        print(
            "[WARNING] Merged CSV not found in working directory. "
            "Run `download_openneuro_ds007554()` and `run_ecg_preprocessing_pipeline()` first."
        )
Displaying hrv_sampen_lmem_pipeline.py.

SyntaxError: invalid syntax (493930552.py, line 2)